# Bloomberg pull smoke test

Usa el kernel **dzlabs venv** (el mismo que `bbgapi_jumpstart`).

Sintaxis oficial:
```python
from bbgapi.query import BlpQuery
bquery = BlpQuery().start()
bquery.bdh(securities='SPX Index', fields=['PX_LAST'],
           start_date='20241211', end_date='20241220')
```

In [ ]:
import os, sys
from pathlib import Path

REPO = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/gh_fwd_screener")
if not REPO.exists():
    REPO = Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("cwd:", Path.cwd())
print("python:", sys.executable)

In [ ]:
# 1) Cliente jumpstart
from bbgapi.query import BlpQuery
bquery = BlpQuery().start()
print("BlpQuery started", bquery)

In [ ]:
# 2) Smoke crudo (mismo patrón que jumpstart)
import pandas as pd

df = bquery.bdh(
    securities="USOSFR10 BGN Curncy",
    fields=["PX_LAST"],
    start_date="20260101",
    end_date=pd.Timestamp.today().strftime("%Y%m%d"),
)
print(type(df), getattr(df, "shape", None))
df.tail()

In [ ]:
# 3) Latest via data.py (usa BlpQuery().start() por dentro)
from data import _securities, _fetch_latest, pull_history_into_store, append_live_bar, load_live_store
from curve_config import BBG_START_DATE, BBG_END_DATE, BBG_FREQ, LIVE_STORE_PATH

secs = _securities()
print(f"n_securities={len(secs)}  config={BBG_START_DATE}→{BBG_END_DATE or 'today'} freq={BBG_FREQ}")
print("sample:", secs[:3], "…", secs[-2:])

latest = _fetch_latest()
print("latest", latest.shape, latest.index[-1])
latest.T.head(8)

In [ ]:
# 4) History corta daily → sofr_live.csv
hist = pull_history_into_store(
    start_date="20260101",
    end_date="",
    freq="1D",
)
print(hist.shape, hist.index[0], "→", hist.index[-1])
hist.iloc[:, :5].tail()

In [ ]:
# 5) Append live print
panel = append_live_bar()
print(panel.shape, "last=", panel.index[-1])
print("on disk", LIVE_STORE_PATH, "n=", len(load_live_store()))
panel.iloc[:, :5].tail(3)

In [ ]:
# 6) Predicción encima del store (sin re-pull)
from run_prediction import run_once
table = run_once(refresh=False)
table[["model", "structure", "nivel", "y_hat_now", "residual", "y_hat_next"]].head(12)